[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hany73-max/PDE-Rover-Engine/blob/main/labs/Lab_6_Mars_Rover_Capstone.ipynb)


In [ ]:
# 🚀 GOOGLE COLAB SETUP
# If you are running this notebook on Google Colab, execute this cell to clone the repo and set paths:
import os, sys
try:
    if 'google.colab' in str(get_ipython()):
        if not os.path.exists('PDE-Rover-Engine'):
            import subprocess
            subprocess.run(['git', 'clone', 'https://github.com/hany73-max/PDE-Rover-Engine.git'], check=True)
        os.chdir('PDE-Rover-Engine')
        if '.' not in sys.path:
            sys.path.insert(0, '.')
        print('[OK] Google Colab setup complete! Connected to PDE-Rover-Engine repository.')
except NameError:
    pass


# 🚀 Lab 6: Capstone — Finite Differences, Red-Black SOR, and Autonomous Mars Navigation
### *Mission Control: Assembling the Complete PDE Path-Planning Engine*
**Course:** Applied Partial Differential Equations & Planetary Robotics  
**Reference:** Integrating Labs 1–5 into [`rover_path_finder/solvers.py`](rover_path_finder/solvers.py) and [`rover_path_finder/planners.py`](rover_path_finder/planners.py)

---
### 🗺️ Full Course Pipeline: How the 5 Lego Bricks (Labs 1–5) Assemble Into the Rover Engine
```
[Lab 1: Terrain Slopes slope_map] ──► [Lab 2: Streamline Paths descend] ──► [Lab 3: Safety Buffers dilate]
                                                                                       │
📍 [Lab 6: Finite-Difference SOR + Neumann Launch] ◄── [Lab 5: Narrow Corridor Decay] ◄── [Lab 4: Laplace Theory _avg]
```

| Building Block (Lab) | Lecture Notes Math (PDF pp. 1–62) | Engineering Problem It Solves on Mars | Real Engine Function (`rover_path_finder/`) |
| :--- | :--- | :--- | :--- |
| **🧱 Brick 1 (Lab 1)** | $p = \frac{\partial z}{\partial x},\; q = \frac{\partial z}{\partial y}$ & Eliminating Constants *(PDF pp. 2–11)* | Computes terrain steepness $\|\nabla z\| = \sqrt{p^2+q^2}$ to flag unsafe crater slopes (`> 0.30`) | [`terrain.slope_map(elev)`](rover_path_finder/terrain.py) |
| **🧱 Brick 2 (Lab 3)** | Non-Linear PDEs & Eikonal Distance $p^2 + q^2 = 1$ *(PDF pp. 17–27)* | Expands every rock/crater outward by the rover's body radius $R$ so wheels never clip rims | [`terrain.dilate(raw, rover_radius)`](rover_path_finder/terrain.py) |
| **🧱 Brick 3 (Lab 4)** | Laplace's Eq. $(D_x^2 + D_y^2)\phi = 0$ & Roots $m = \pm i$ *(PDF pp. 28–51)* | Creates a smooth harmonic potential $\phi(x,y)$ guaranteed to have **zero local minima** | [`solvers._avg(phi)`](rover_path_finder/solvers.py) |
| **🧱 Brick 4 (Lab 5)** | Reducible Factor $(aD_x+bD_y+c)z=0 \implies e^{-(c/a)x}$ *(PDF pp. 52–62)* | Explains exponential corridor decay $e^{-\pi x/w}$ under Dirichlet walls and switches to Neumann walls | [`planners.neumann_potential()`](rover_path_finder/planners.py) |
| **🧱 Brick 5 (Lab 2)** | Lagrange's Characteristics $\frac{dx}{P} = \frac{dy}{Q} = \frac{dz}{R}$ *(PDF pp. 12–17)* | Traces the steepest-descent streamline $(-\phi_x, -\phi_y)$ from `start` to `goal` | [`planners.descend(phi, hazard, start, goal)`](rover_path_finder/planners.py) |

---
### 🧭 The Rosetta Stone: Connecting Math to `rover_path_finder/`

| 1. Mathematical Concept | 2. Physical & Geometric Meaning | 3. Real Code in `rover_path_finder/` |
| :--- | :--- | :--- |
| $\nabla^2\phi = \frac{\partial^2\phi}{\partial x^2} + \frac{\partial^2\phi}{\partial y^2} = 0$ | **Laplace's Equation (Lab 4):** smooth harmonic potential with zero interior local minima | Governing PDE solved in [`solvers.py`](rover_path_finder/solvers.py) |
| $\phi_{i,j} = \frac{1}{4}(\phi_{i+1,j} + \phi_{i-1,j} + \phi_{i,j+1} + \phi_{i,j-1})$ | **5-Point Finite-Difference Stencil:** discrete average over 4 orthogonal neighbors | [`_avg(phi)`](rover_path_finder/solvers.py) in `solvers.py` |
| $\frac{\partial\phi}{\partial n} = 0$ on obstacle walls | **Insulated Neumann BCs (Lab 5):** flow bends around rocks without decaying in narrow corridors | Dividing free-neighbor sum by active count `cnt` in [`sor_neumann`](rover_path_finder/solvers.py) |
| $\omega_{\text{opt}} = \frac{2}{1 + \sin(\pi/N)}$ | **Successive Over-Relaxation (SOR):** amplifies each update to accelerate convergence | [`optimal_omega(N)`](rover_path_finder/solvers.py) in `solvers.py` |
| $\frac{dx}{-\phi_x} = \frac{dy}{-\phi_y}$ | **Lagrange Characteristic Streamline (Lab 2):** steepest descent trajectory from `start` to `goal` | [`descend(phi, hazard, start, goal)`](rover_path_finder/planners.py) |

---
## 1. Why Do We Discretize Laplace's Equation onto a Grid?

In Labs 1–5, we solved PDEs analytically on paper. On Mars, however:
* Our synthetic testbed `make_terrain(N=100)` has **$100 \times 100 = 10,000$ grid nodes**, and the real NASA HiRISE DEM (`kasei.IMG` in `main.py`) scales up to **$160 \times 160 = 25,600$ nodes** with thousands of irregular crater walls!
* No closed-form paper formula can satisfy $\frac{\partial\phi}{\partial n} = 0$ on thousands of jagged rock pixels simultaneously.
* Instead, we discretize $\nabla^2\phi = 0$ using the **Finite Difference Method (5-Point Stencil)** and solve the resulting linear system iteratively using **Red-Black Successive Over-Relaxation (SOR)**.

> 📌 **Honest Syllabus Note:** While analytical solutions to Laplace's equation $(D_x^2 + D_y^2)z = 0$ are covered in Chapter 1.4 of the lecture notes (Notes p. 39 / PDF p. 33), **finite-difference discretization, the Discrete Maximum Principle, and SOR** are computational enrichment topics that bridge your lecture notes to real-world engineering software.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from labs.tests import test_lab6

%matplotlib inline
print("Lab 6 Capstone Initialized! Mission Launch Sequence Ready.")


---
## 2. Step-by-Step Mathematical Derivation: Taylor Series & The Discrete Maximum Principle

### Part A: Deriving the 5-Point Stencil from Taylor Series
Expanding $\phi(x \pm h, y)$ around node $(i, j)$ via Taylor's theorem gives:
$$\phi_{i+1, j} = \phi_{i,j} + h \left.\frac{\partial\phi}{\partial x}\right|_{i,j} + \frac{h^2}{2}\left.\frac{\partial^2\phi}{\partial x^2}\right|_{i,j} + \frac{h^3}{6}\left.\frac{\partial^3\phi}{\partial x^3}\right|_{i,j} + \mathcal{O}(h^4)$$
$$\phi_{i-1, j} = \phi_{i,j} - h \left.\frac{\partial\phi}{\partial x}\right|_{i,j} + \frac{h^2}{2}\left.\frac{\partial^2\phi}{\partial x^2}\right|_{i,j} - \frac{h^3}{6}\left.\frac{\partial^3\phi}{\partial x^3}\right|_{i,j} + \mathcal{O}(h^4)$$

Adding the two equations cancels all odd-order derivatives ($\partial\phi/\partial x$ and $\partial^3\phi/\partial x^3$):
$$\left.\frac{\partial^2\phi}{\partial x^2}\right|_{i,j} \approx \frac{\phi_{i+1, j} - 2\phi_{i,j} + \phi_{i-1, j}}{h^2}, \qquad \left.\frac{\partial^2\phi}{\partial y^2}\right|_{i,j} \approx \frac{\phi_{i, j+1} - 2\phi_{i,j} + \phi_{i, j-1}}{h^2}$$

Substituting into Laplace's equation $\frac{\partial^2\phi}{\partial x^2} + \frac{\partial^2\phi}{\partial y^2} = 0$ and multiplying by $h^2$:
$$(\phi_{i+1, j} + \phi_{i-1, j} + \phi_{i, j+1} + \phi_{i, j-1}) - 4\phi_{i,j} = 0 \implies \boxed{\phi_{i,j} = \frac{1}{4}\left(\phi_{i+1, j} + \phi_{i-1, j} + \phi_{i, j+1} + \phi_{i, j-1}\right)}$$

---
### Part B: 1-Line Proof of the Discrete Maximum Principle (Why Local Traps Are Impossible)
Why is it 100% impossible for the rover to get stuck in an interior local minimum?
* **Proof by contradiction:** Suppose an interior free node $(i, j)$ were a strict local minimum, meaning its value is strictly less than all four of its neighbors:
  $$\phi_{i,j} < \phi_{\text{North}}, \quad \phi_{i,j} < \phi_{\text{South}}, \quad \phi_{i,j} < \phi_{\text{East}}, \quad \phi_{i,j} < \phi_{\text{West}}$$
* Summing these four strict inequalities and dividing by 4 gives:
  $$\phi_{i,j} < \frac{1}{4}\left(\phi_{\text{North}} + \phi_{\text{South}} + \phi_{\text{East}} + \phi_{\text{West}}\right)$$
* Which directly contradicts Laplace's stencil equation $\phi_{i,j} = \frac{1}{4}(\phi_N + \phi_S + \phi_E + \phi_W)$!
* Therefore, **no interior free node can ever be a local minimum**—the only minimum on the entire map is the goal sink ($\phi = 0$)!

---
### Part C: Zero-Flux Neumann Obstacle Walls ($\frac{\partial\phi}{\partial n} = 0$)
When an adjacent pixel is an obstacle (`free == False`), no current may flow into it. In [`solvers.sor_neumann`](rover_path_finder/solvers.py), we mask out obstacle values (`pf = phi * f`) and divide by the **number of active free neighbors** `cnt` (which is 3 along a straight wall, or 2 in a corner):
$$\phi_{i,j} = \frac{\sum_{\text{free neighbors}} \phi_{\text{neighbor}}}{\text{cnt}_{i,j}}$$


---
## 3. 📝 Paper & Pen Challenge 1 (4-Node Hand Calculation — Jacobi Sweep 1)

Let's compute **Sweep 1 of the synchronous Jacobi method** by hand on a $4 \times 4$ grid whose 4 interior cells $(A, B, C, D)$ start at $\phi^{(0)} = 0.0$:

```
             Top Boundary (Fixed Dirichlet: phi = 1.0)
         +--------+--------+--------+--------+
         |  1.0   |  1.0   |  1.0   |  1.0   |
         +--------+--------+--------+--------+
Left     |  0.5   | A(0.0) | B(0.0) |  0.5   | Right Boundary
Boundary +--------+--------+--------+--------+ (Fixed Dirichlet:
(Fixed:  |  0.5   | C(0.0) | D(0.0) |  0.5   |  phi = 0.5)
phi=0.5) +--------+--------+--------+--------+
         |  0.0   |  0.0   |  0.0   |  0.0   |
         +--------+--------+--------+--------+
            Bottom Boundary (Fixed Dirichlet: phi = 0.0)
```

> 📌 **Important:** In a **Jacobi sweep**, all new values $\phi^{(1)}$ are computed simultaneously from the initial values $\phi^{(0)} = 0.0$:
> * Initial neighbors of **Cell A**: North $= 1.0$, South ($C^{(0)}$) $= 0.0$, West $= 0.5$, East ($B^{(0)}$) $= 0.0$.
> * Initial neighbors of **Cell B**: North $= 1.0$, South ($D^{(0)}$) $= 0.0$, West ($A^{(0)}$) $= 0.0$, East $= 0.5$.
> *(Note: If you used sequential Gauss-Seidel instead, Cell B would use the newly updated $A^{(1)} = 0.375$ to get $0.46875$; here we ask for the **synchronous Jacobi** values).*

Compute $\phi_A^{(1)}$ and $\phi_B^{(1)}$ and enter them below:

<details>
<summary>💡 <b>Hint (Averaging the 4 Initial Neighbors)</b></summary>

For Cell A, $(1.0 + 0.0 + 0.5 + 0.0) / 4 = 1.5 / 4 = 0.375$. By left-right symmetry, Cell B has the exact same initial neighbor sum!
</details>


In [ ]:
# GRADED CELL: 4-Node Hand Calculation (Jacobi Sweep 1)
### START CODE HERE ###
phi_A = 0.375
phi_B = 0.375
### END CODE HERE ###

# Run numerical verification test:
test_lab6.check_paper_challenge_1(phi_A, phi_B)


---
## 4. 🚀 Paper & Pen Challenge 2 (Optimal SOR Factor $\omega_{\text{opt}}$ — `solvers.optimal_omega`)

Standard Jacobi relaxation is slow on large grids because boundary information propagates by only 1 pixel per sweep.
**Successive Over-Relaxation (SOR)** accelerates convergence dramatically by over-shooting the Gauss-Seidel average by a factor $\omega \in (1, 2)$:
$$\phi_{i,j}^{\text{new}} = (1 - \omega)\,\phi_{i,j}^{\text{old}} + \omega\,\text{avg}_{i,j}$$

By Young's Theorem (implemented in [`solvers.optimal_omega(N)`](rover_path_finder/solvers.py)), the theoretical optimal relaxation parameter on an $N \times N$ grid is:
$$\omega_{\text{opt}} = \frac{2}{1 + \sin\left(\frac{\pi}{N}\right)}$$

Compute $\omega_{\text{opt}}$ for $N = 16$:

<details>
<summary>💡 <b>Hint (Evaluating Young's Formula)</b></summary>

Substitute $N = 16$ into `2.0 / (1.0 + np.sin(np.pi / 16))` (which evaluates to approximately `1.6735`).
</details>


In [ ]:
# GRADED CELL: Optimal Omega for N=16
### START CODE HERE ###
omega_16 = 2.0 / (1.0 + np.sin(np.pi / 16))
### END CODE HERE ###

# Run numerical verification test:
test_lab6.check_paper_challenge_2(omega_16)


---
## 5. 💻 Python Implementation: Red-Black SOR with Zero-Flux Neumann Walls

Now implement `sor_neumann_step`, the core solver inside [`rover_path_finder/solvers.py`](rover_path_finder/solvers.py):
1. **Red-Black Checkerboard Splitting (`masks`):** Updates all Red cells (`(x + y) % 2 == 0`) simultaneously using vectorized NumPy rolls, then updates all Black cells (`(x + y) % 2 == 1`) using the freshly updated Red neighbors.
2. **Neumann Zero-Flux Masking (`pf = phi * f` divided by `cnt`):** Prevents potential from leaking into obstacle pixels.
3. **SOR Update:** `phi[mk] = (1 - omega) * old + omega * avg[mk]`.


In [ ]:
# GRADED FUNCTION: sor_neumann_step
def sor_neumann_step(phi, fixed, free, omega=1.5, tol=1e-5, maxit=5000):
    # Red-Black SOR solver with zero-flux Neumann crater walls
    phi = phi.copy()
    n, m = phi.shape
    f = free.astype(float)
    
    # Active free neighbor count (1 to 4) for Neumann boundary condition d(phi)/dn = 0
    cnt = sum(np.roll(f, s, a) for a in (0, 1) for s in (1, -1))
    cnt = np.maximum(cnt, 1.0)
    
    yy, xx = np.mgrid[0:n, 0:m]
    parity = (xx + yy) % 2
    masks = [(parity == c) & free & ~fixed for c in (0, 1)]
    phi[~free] = 0.0
    hist = []
    
    ### START CODE HERE ###
    for _ in range(maxit):
        d = 0.0
        for mk in masks:
            pf = phi * f
            avg = (
                np.roll(pf, 1, 0) + np.roll(pf, -1, 0)
                + np.roll(pf, 1, 1) + np.roll(pf, -1, 1)
            ) / cnt
            old = phi[mk]
            phi[mk] = (1 - omega) * old + omega * avg[mk]
            d = max(d, float(np.abs(phi[mk] - old).max()))
        hist.append(d)
        if d < tol:
            break
    ### END CODE HERE ###
    
    return phi, hist

# Run automated unit test:
test_lab6.test_sor_neumann_step(sor_neumann_step)


---
## 6. 🌉 Classical Artificial Potential Fields (APF) vs. Laplace Harmonic Potential

In classical robotics, **Artificial Potential Fields (`planners.apf_potential`)** add an attractive bowl toward the goal and a local repulsive barrier around obstacles:
$$U(\mathbf{x}) = U_{\text{att}}(\mathbf{x}) + U_{\text{rep}}(\mathbf{x})$$

| Feature | Classical APF (`apf_potential`) | Laplace PDE (`neumann_potential`) |
| :--- | :--- | :--- |
| **Governing Equation** | Algebraic sum $U_{\text{att}} + U_{\text{rep}}$ ($\nabla^2 U \neq 0$) | Elliptic PDE $\nabla^2\phi = 0$ with $\frac{\partial\phi}{\partial n} = 0$ |
| **Concave / U-Shaped Obstacles** | Attractive pull and repulsive push cancel out ($\nabla U = \mathbf{0}$), **trapping the rover** | **Zero interior local minima** (Discrete Maximum Principle); streamlines bend cleanly around walls |

---
## 7. 🎨 Grand Mission Finale: Autonomous Navigation on Martian Terrain (`100x100`)

Run the cell below to generate synthetic Martian terrain (`10,000` nodes), solve Laplace's equation with insulated Neumann boundaries (`tol=1e-6`), and trace the rover's trajectory (`p[0]` = $x$, `p[1]` = $y$) cleanly around every red hazard zone!


In [ ]:
from rover_path_finder.terrain import make_terrain
from rover_path_finder.planners import neumann_potential, descend, apf_potential, trap_map

# 1. Generate 100x100 Martian terrain with rover_radius=3 safety dilation
t_data = make_terrain(N=100, seed=42, rover_radius=3)
elev, raw_hazards, hazards = t_data["h"], t_data["raw"], t_data["hazard"]
start = t_data["start"]
goal = t_data["goal"]

# 2. Solve Laplace's Equation with Neumann zero-flux walls (tol=1e-6 for fast execution)
phi, sweeps, elapsed = neumann_potential(hazards, start=start, goal=goal, tol=1e-6)

# 3. Trace characteristic streamline via steepest descent
path, reached = descend(phi, hazards, start=start, goal=goal)

# 4. Plot Navigation Field & Path (p[0] is x=col, p[1] is y=row)
fig, ax = plt.subplots(figsize=(9.5, 7.5))
cs = ax.contourf(phi, levels=30, cmap='viridis')
plt.colorbar(cs, ax=ax, label=r"Harmonic Potential $\phi(x, y)$")
ax.contour(hazards, levels=[0.5], colors='red', linewidths=1.5)

px = [p[0] for p in path]
py = [p[1] for p in path]
ax.plot(px, py, color='cyan', linewidth=3, label="Rover Trajectory (Laplace Streamline)")
ax.plot(start[0], start[1], 'go', markersize=10, label=r"Start ($\phi=1$)")
ax.plot(goal[0], goal[1], 'r*', markersize=14, label=r"Goal ($\phi=0$)")

ax.set_title("Ares Rover Navigation: Laplace Streamlines Around Martian Hazards", fontsize=13)
ax.set_xlabel("x (column index)")
ax.set_ylabel("y (row index)")
ax.legend(loc='upper left')
plt.show()

print(f"Mission Status: Reached Goal = {reached} | Path Length = {len(path)} steps | SOR Sweeps = {sweeps} ({elapsed:.2f}s)")


---
## 8. 🪤 The Ultimate Benchmark: Escaping the U-Shaped Box Canyon (`trap_map`)

Run the cell below to compare **Classical APF** (left, permanently stuck inside the U-shaped canyon at $(55, 50)$) against **Laplace PDE with Neumann Walls** (right, smoothly navigating around the canyon wall to reach the goal $(90, 50)$):


In [ ]:
u_trap, trap_start, trap_goal = trap_map(N=100)

# 1. Classical APF
apf_field = apf_potential(u_trap, goal=trap_goal)
apf_path, apf_reached = descend(apf_field, u_trap, start=trap_start, goal=trap_goal, maxsteps=200)

# 2. Laplace PDE with Neumann BCs (tol=1e-6)
laplace_field, trap_sweeps, trap_time = neumann_potential(u_trap, start=trap_start, goal=trap_goal, tol=1e-6)
laplace_path, laplace_reached = descend(laplace_field, u_trap, start=trap_start, goal=trap_goal, maxsteps=400)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Classical APF Plot
axes[0].contourf(apf_field, levels=25, cmap='magma')
axes[0].contour(u_trap, levels=[0.5], colors='white', linewidths=1.5)
axes[0].plot([p[0] for p in apf_path], [p[1] for p in apf_path], 'r-', linewidth=3, label="APF Path (Stuck)")
axes[0].plot(trap_start[0], trap_start[1], 'go', markersize=9, label="Start (20, 50)")
axes[0].plot(trap_goal[0], trap_goal[1], 'y*', markersize=13, label="Goal (90, 50)")
axes[0].set_title("Classical Robotics (APF): TRAPPED in Local Minimum", fontsize=12)
axes[0].set_xlabel("x")
axes[0].set_ylabel("y")
axes[0].legend(loc="upper left")

# Laplace PDE Plot
axes[1].contourf(laplace_field, levels=25, cmap='viridis')
axes[1].contour(u_trap, levels=[0.5], colors='red', linewidths=1.5)
axes[1].plot([p[0] for p in laplace_path], [p[1] for p in laplace_path], 'c-', linewidth=3, label="Laplace Path (Escapes)")
axes[1].plot(trap_start[0], trap_start[1], 'go', markersize=9, label="Start (20, 50)")
axes[1].plot(trap_goal[0], trap_goal[1], 'y*', markersize=13, label="Goal (90, 50)")
axes[1].set_title("Laplace PDE: Maneuvers Cleanly Around the U-Trap", fontsize=12)
axes[1].set_xlabel("x")
axes[1].set_ylabel("y")
axes[1].legend(loc="upper left")

plt.tight_layout()
plt.show()

print("Benchmark Results:")
print(f"  • Classical APF: Reached target? {apf_reached} (Permanently trapped inside U-shaped wall)")
print(f"  • Laplace PDE  : Reached target? {laplace_reached} (Safely escaped the trap in {len(laplace_path)} steps, {trap_time:.2f}s!)")


## 🏆 Course Graduation

**Congratulations on completing the Applied PDE & Rover Navigation Lab Suite!**

You have connected every topic in your PDE lecture notes to a working autonomous navigation engine:
1. **Lab 1 (Elimination of Arbitrary Constants):** Connected partial derivatives $(p, q)$ to [`terrain.slope_map`](rover_path_finder/terrain.py).
2. **Lab 2 (Lagrange's Method $Pp+Qq=R$):** Connected characteristic curves to [`planners.descend`](rover_path_finder/planners.py).
3. **Lab 3 (Non-Linear PDEs & Charpit's Method):** Connected the Eikonal distance equation $p^2+q^2=1$ to [`terrain.dilate`](rover_path_finder/terrain.py).
4. **Lab 4 (Higher-Order Homogeneous Operators):** Solved Laplace's equation $(D_x^2+D_y^2)z=0$ and verified the 4-neighbor average [`solvers._avg`](rover_path_finder/solvers.py).
5. **Lab 5 (Non-Homogeneous Operators):** Solved linear factors $(aD_x+bD_y+c)z=0$ and proved why Dirichlet walls decay exponentially in narrow corridors.
6. **Lab 6 (Finite Differences & Red-Black SOR):** Built `sor_neumann_step` with insulated Neumann walls and escaped U-shaped local-minimum traps!
